# Clustering the result of a MCA with K-means

We experiment here with a way of detecting specific profiles in the population using K-means clustering on the results of the MCA.

* [Clustering](https://en.wikipedia.org/wiki/Cluster_analysis) (Wikidata)
* [K-means](https://en.wikipedia.org/wiki/K-means_clustering) (Wikidata)

In [ ]:
import numpy as np
import pandas as pd

import sqlite3 as sql

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import seaborn as sns
import pprint as pprint

import networkx as nx
import itertools

import scipy.cluster.hierarchy as sch
from scipy.cluster.hierarchy import fcluster
from scipy.spatial.distance import cdist

import fanalysis.ca as fa 

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from prince import MCA


In [ ]:
### Importer un module de fonctions crées ad hoc
##  ATTENTION : le fichier 'sparql_functions.py' doit se trouver 
#   dans un dossier qui se situe dans le chemin ('path') de recherche
#   vu par le présent carnet Jupyter afin que
#   l'importation fonctionne correctement

import sys
from importlib import reload


# Add parent directory to the path
sys.path.insert(0, '..')

### If you want to add the parent-parent directory,
sys.path.insert(0, '../..')



In [ ]:
import bivariate_library as bl
import correspondence_analysis_library as cal
import cluster_functions as cf


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
### Use this command to reload the functions if modified
print(reload(cf))

## Create a dataframe with the data to be analysed

In this notebook, we use the data produced in the [multiple correspondence analysis (MCA) notebook ](da5_MCA.ipynb), which was prepared and exported into this file [this file](../da_data/da5-MCA-clusters.csv) after the sparse data was aggregated.



In [ ]:
file_address='../da_data/da5-MCA-clusters.csv'
df_pm = pd.read_csv(file_address)
df_pm.head(3)

In [ ]:
print(df_pm.columns.to_list())

In [ ]:
### Rename columns to have shorter labels
df_pm.columns=['person_uri',
 'labelPer',
 'birthYear',
 'gender',
 'labelPlace',
 'geometry',
 'REGION',
 'NAME_ENGL',
 'country',
 'per_activ',
 'pk_person_features',
 'occup_m',
 'occup_s',
 'empl']

In [ ]:
### Inspect the dataframe and 
# notably if there are missing values
df_pm.info()

In [ ]:
### change display options for jupyter notebooks
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
# Reset to default settings if needed later
# pd.reset_option('display.max_columns')

## Multiple factor analysis (MCA)

We only consider for MCA the coded features of persons, transformed into qualitative categories 

We don't use the periods for defining profiles, our research question is about inspecting the correlation between profiles and periods, and identify specific profiles for specific periods

In [ ]:
### Define the variables that will be used for analysing the population profiles
df_actives = df_pm[['gender', 'country', 'occup_m', 'occup_s', 'empl']].copy(deep=True)
print(df_actives.columns)
df_actives.head(2)

#### Process in two steps

First fit, then transform.

The mca variable will be used to project illustrative variables, in our case periods of activity

In [ ]:
### MCA with benzecri correction: whole intertia in 5 axis
# Equivalent to fit_transform but in to steps
mca = MCA(n_components=70, n_iter=10, correction='benzecri',random_state=42)
mca.fit(df_actives)

In [ ]:
### Benzecri correction and approximation allows to summarize 
# the most relevant aspects of the variance in five axes (in this case)
mca.eigenvalues_summary.head(15)

### Keep 3 components

In [ ]:
### MCA with benzecri correction: whole intertia in 10 axis
# Equivalent to fit_transform but in to steps
mca = MCA(n_components=3, n_iter=10, correction='benzecri',random_state=42)
mca.fit(df_actives)

In [ ]:
### Benzecri correction and approximation allows to summarize 
# the most relevant aspects of the variance in five axes (in this case)
mca.eigenvalues_summary.head(15)

In [ ]:
### Plot the part of variance (eigenvalue) of each dimension (or table) 
mca.scree_plot()
### saved as png, not visible on github


### Show image scree plot (visible on github)

img_address='doc_images/mca_prince_scree_plot.png'
# Read the image
img = mpimg.imread(img_address)

# Display the image
plt.imshow(img)
plt.axis('off')  # Hide axes
plt.show()

In [ ]:
### categories coordinates, i.e. columns in the complete disjoint table
print(len(mca.column_coordinates(df_actives)))
dfcc = mca.column_coordinates(df_actives)
dfcc.head()

In [ ]:
### individual coordinates (rows)
# The same result as using the method 'fit_transform'
print(len(mca.row_coordinates(df_actives)))
dfrc = mca.row_coordinates(df_actives)
dfrc.head()

In [ ]:
### Equivalent to mca.row_coordinates(df_actives)
X_mca=mca.transform(df_actives)

In [ ]:
X_mca.head(2)

### Plot categories, first two axis

In [ ]:
ax1=mca.eigenvalues_summary.iloc[0, 1]
ax2=mca.eigenvalues_summary.iloc[1, 1]

In [ ]:

fig, ax = plt.subplots(figsize=(18, 18))

ax.scatter(dfcc[0], dfcc[1], s=30, alpha=0.7)

# Label each point with its index
for label, x, y in zip(dfcc.index, dfcc[0], dfcc[1]):
    ax.annotate(label, (x, y), fontsize=10, ha='right', va='bottom')

ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')
ax.axvline(0, color='grey', linewidth=0.5, linestyle='--')
ax.set_xlabel(f"Component 1 ({ax1})")
ax.set_ylabel(f"Component 2 ({ax2})")
ax.set_title("MCA – Axis 1 & 2")

plt.tight_layout()
plt.show()

### Calculate position of illustrative variable 'per_activ'

We don't use the periods for defining profiles, our research question is about inspecting the correlation between profiles and periods, and identify specific profiles for specific periods

In [ ]:
df_illustrative = df_pm[['per_activ']].copy(deep=True)
df_illustrative.head(2)

In [ ]:
df_per_activ=pd.DataFrame(df_pm.groupby('per_activ').size())
# df_per_activ.index=df_per_activ.index.map(lambda x: f'per_activ__{x}')
df_per_activ.columns=['number']
df_per_activ

In [ ]:
### This script situates the illustrative variable 'per_activ'
# at the mean position in the geometric space using the coordinates 
# of all the individuals

row_coords = mca.row_coordinates(df_actives)

# 3. Select the single illustrative variable column
illus_var = df_illustrative['per_activ'] # Replace with actual column name

# 4. Calculate position for each category in that variable
supp_positions = {}

for category in illus_var.dropna().unique():
    # Find indices of individuals who have this category
    mask = illus_var == category
    
    # Calculate the barycenter (mean) of their row coordinates
    # This is the position of the supplementary category
    centroid = row_coords[mask].mean(axis=0)
    
    supp_positions[category] = centroid.values

# 5. Convert to DataFrame for easy use/plotting
df_per_activ_pos = pd.DataFrame(
    supp_positions, 
    index=[i for i in range(row_coords.shape[1])]
).T  # Transpose so rows are categories, columns are dimensions

df_per_activ_pos

In [ ]:
### We add the label and the number of individuals to the mean position
df_per_activ=df_per_activ.join(df_per_activ_pos)

In [ ]:
df_per_activ.head(2)

In [ ]:
### dimensions
d1=1 # 0
d2=2 # 1

# Create the plot
fig, ax = plt.subplots(figsize=(20, 20))

# Plot Active Variables (Blue/Grey default)
ax.scatter(dfcc[d1], dfcc[d2], s=30, alpha=0.7, c='blue', label='Active Variables', zorder=3)

# Label Active Variables
for label, x, y in zip(dfcc.index, dfcc[d1], dfcc[d2]):
    ax.annotate(label, (x, y), fontsize=10, ha='right', va='bottom', color='blue')

# Plot Illustrative Variables (Red)
if not df_per_activ.empty:
    ax.scatter(df_per_activ[d1], df_per_activ[d2], 
               s=30, alpha=0.8, c='red', marker='s', label='Illustrative Variables', zorder=4)
    
    # Label Illustrative Variables (optional: adjust font size/color to distinguish)
    for label, x, y in zip(df_per_activ.index, df_per_activ[d1], df_per_activ[d2]):
        ax.annotate(label, (x, y), fontsize=10, ha='left', va='top', color='red', fontweight='bold')

# Add reference lines
ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')
ax.axvline(0, color='grey', linewidth=0.5, linestyle='--')

# Set labels and title
ax.set_xlabel(f"Component 1 ({ax1})") # Formatting percentage if applicable
ax.set_ylabel(f"Component 2 ({ax2})")
ax.set_title("MCA – Axis 1 & 2 (Active vs Illustrative)")

# Add legend to distinguish groups
ax.legend(loc='best')

# Save the figure
img_address='doc_images/mca_prince_with_periods.png'
plt.savefig(img_address, dpi=150, bbox_inches='tight')

plt.tight_layout()
plt.show()

### Ascendant hierarchical classification


We observe that some variation is present in the population but it is hard to identify using MCA the relevant profiles.

We therefore will use the K-means machine learning methodology to create clusters using the positions in the geometric space calculated with MCA.


* We first inspect and choose the optimal clustering level using a hierarchical classificaion
* We the apply two other techniques to decide the optimal clustering level: Elbow vs Silouette scores

In [ ]:
Z = sch.linkage(X_mca, method='ward')

# --- Dendrogram with cut lines ---
fig, ax = plt.subplots(figsize=(15, 10))

sch.dendrogram(Z, labels=df_pm.index.tolist(), leaf_rotation=90, 
               leaf_font_size=6, ax=ax)

dfs = {}

# Add horizontal cut lines for  clusters  | 32 64
colors = {'16 clusters': 'red', '48 clusters': 'green', '64 clusters': 'blue'}
for clusters_n, color in zip([16, 48, 56], ['red', 'green', 'blue']):
    cut_height = sch.dendrogram(Z, no_plot=True)['dcoord']

    labels = fcluster(Z, t=clusters_n, criterion='maxclust')

    dfs[clusters_n] = df_pm.assign(cluster=labels)

    # get the threshold height for this number of clusters
    threshold = sorted(Z[:, 2], reverse=True)[clusters_n - 1]
    ax.axhline(y=threshold, color=color, linestyle='--', 
               label=f'{clusters_n} clusters')

ax.set_title("Hierarchical Ascendant Classification (HAC)")
ax.set_xlabel("Individuals")
ax.set_ylabel("Distance")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
### Test best clustering k
# we test from 2x8 to 9x8, althoug we'll use 64

# 2. Define cluster range
k_range = range(16, 73)

# 3. Calculate Inertia and Silhouette Scores
inertias = []
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_mca)
    
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_mca, labels))

# 4. Plot results
fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.plot(k_range, inertias, 'bo-', label='Inertia (Elbow)')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia', color='b')
ax1.tick_params(axis='y', labelcolor='b')

ax2 = ax1.twinx()
ax2.plot(k_range, silhouette_scores, 'rs-', label='Silhouette Score')
ax2.set_ylabel('Silhouette Score', color='r')
ax2.tick_params(axis='y', labelcolor='r')

plt.title('Optimal Clusters: Elbow vs Silhouette (k=16 to 72)')
fig.tight_layout()
plt.show()

# 5. Identify best k
best_k_silhouette = k_range[np.argmax(silhouette_scores)]
print(f"Best k by Silhouette: {best_k_silhouette} (Score: {max(silhouette_scores):.3f})")

In [ ]:
print([s for s in silhouette_scores][40:55])

In [ ]:
i=range(30,55)
v=[s for s in silhouette_scores][30:55]
a=plt.bar(i, v)

## Define clusters with same number as K-mode to compare


We observe that there is no clear distinction of profiles in the whole population. 

The elbow could be situated at 30-32 clusters but further splitting produces even more precise results, indicating the present of smaller different profiles.

We therefore take the heuristic stance to inspect three numbers of clusters that are multiples of our 8 periods: 16, 32, 64. 

We will the observe if the splitting allow to detect some more clear correlation with periods



In [ ]:
### choose your number of clusters

k=32

In [ ]:
### We activate the machine learning object

kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)

In [ ]:
### We activate the machine learning process that creates the clusters
#  X_mca: coordinates of individuals/rows, cf. above
kmeans.fit(X_mca)

In [ ]:
### Assign cluster labels to the coordinates dataframe
# each row represents an individual
X_mca["cluster"] = kmeans.labels_

In [ ]:
### Result of clustering
# The index is the individuals' index
# The cluster label (as integer) is added to the coordinates in the MCA space
print('Number of rows:', len(X_mca))
X_mca.head()

In [ ]:
### We add the cluster to the categories table of the active variables
# Take only the first 5 columns and exclude the new 'cluster' column
clusters = kmeans.fit_predict(X_mca.iloc[:,:10])
df_actives['cluster']=clusters

In [ ]:
print('Number of rows:', len(df_actives))
print(list(df_actives.columns))
df_actives.head()

### Get the Cluster Profiles

Get the geometric center of coordinates of the cluster and join with the mode, i.e. the most frequent category set in the cluster

In [ ]:
# Get the numeric centroid (average of dimensions in clusters)
#numeric_centroids = pd.DataFrame(kmeans.cluster_centers_, columns=['dim_1', 'dim_2', 'dim_3', 'dim_4','dim_5','dim_6', 'dim_7', 'dim_8', 'dim_9','dim_10'])
numeric_centroids = pd.DataFrame(kmeans.cluster_centers_)
numeric_centroids.index.name = 'Cluster_ID'
numeric_centroids.head()

In [ ]:
### Calculate the categorical profile (Mode of each variable per cluster)
# We use here the mode, i.e. the most frequent set in each cluster
categorical_centroids = df_actives.groupby('cluster').agg(lambda x: x.mode()[0])
categorical_centroids.head()

In [ ]:
### size of the clusters
dfg = pd.DataFrame(df_actives.groupby(by=['cluster']).size())
dfg.columns=['number_in_cl']
dfg.sort_values(by='number_in_cl', ascending=False).head()

In [ ]:
ax=dfg.sort_values(by='number_in_cl', ascending=False).plot(kind='bar',
                                figsize=(8,3),   rot=45, legend=False)

# Optional: Adjust alignment and layout for better appearance
plt.xticks(ha='right', size=8)  # Aligns the end of the text with the tick
plt.tight_layout()      # Prevents labels from being cut off
plt.show()

In [ ]:
### Join to cluster modes the number of individuals in the cluster 
# Beware: not all share the mode !
categorical_centroids=categorical_centroids.join(dfg)

In [ ]:
categorical_centroids.sort_values('number_in_cl', ascending=False)

In [ ]:
df_vact_g=pd.DataFrame(df_pm.groupby(by=['gender', 'country', 'occup_m', 'occup_s', 'empl']).size())
df_vact_g.columns=['number']
df_vact_g=df_vact_g.reset_index()
df_vact_g.sort_values(by='number', ascending=False).iloc[:64]

In [ ]:
df_vact_g=pd.DataFrame(df_pm.groupby(by=['gender', 'country', 'occup_m', 'occup_s', 'empl']).size())
df_vact_g.columns=['number']
df_vact_g=df_vact_g.reset_index()
#df_vact_g[df_vact_g.gender=='female'].sort_values(by='number', ascending=False).iloc[:64]
df_vact_g_f=df_vact_g[(df_vact_g.gender=='female') & (df_vact_g. number > 2)].sort_values(by='country')
print(len(df_vact_g_f))
df_vact_g_f

In [ ]:
pprint.pprint([list(e) for e in df_vact_g[df_vact_g.gender=='female'].sort_values(by='number', ascending=False).iloc[:64].values])

In [ ]:
# Merge the cluster modes with the nummerical centroids 
cluster_profiles = pd.concat([categorical_centroids,numeric_centroids], axis=1)

In [ ]:
### Inspect
cluster_profiles.head(3)

In [ ]:
### Create a label with the mode categories
cluster_profiles['label'] = cluster_profiles.apply(
    lambda x: str(x.name) + "_" + "_".join(x[['gender', 'country', 'occup_m','occup_s', 'empl']].astype(str)), 
    axis=1
)
cluster_profiles[['label', 'number_in_cl']].head(3)

In [ ]:
### Inspect the already created illustrative variable 'per_act'
df_per_activ

### Plot the cluster profiles

In [ ]:
### Get the inertia proportion for the first two axis
ax1=mca.eigenvalues_summary.iloc[0, 1]
ax2=mca.eigenvalues_summary.iloc[1, 1]

In [ ]:
# Get coordinates for illustrative variables
# Note: Ensure 'df_illustrative' contains the column names of your illustrative variables
df_illust_coords = mca.column_coordinates(df_illustrative)

# Create the plot
fig, ax = plt.subplots(figsize=(20, 20))

# Plot Active Variables (Blue/Grey default)
ax.scatter(cluster_profiles[0], cluster_profiles[1], 
           s=cluster_profiles['number_in_cl'], alpha=0.7, c='blue', label='Active Variables', zorder=3)

# Label Active Variables
for label, x, y in zip(cluster_profiles.label, cluster_profiles[0], cluster_profiles[1]):
    ax.annotate(label, (x, y), fontsize=10, ha='right', va='bottom', color='blue')

# Plot Illustrative Variables (Red)
if not df_per_activ.empty:
    ax.scatter(df_per_activ[1], df_per_activ[2], 
               s=df_per_activ['number']/5, alpha=0.8, c='red', marker='s', label='Illustrative Variables', zorder=4)
    
    # Label Illustrative Variables (optional: adjust font size/color to distinguish)
    for label, x, y in zip(df_per_activ.index, df_per_activ[1], df_per_activ[2]):
        ax.annotate(label, (x, y), fontsize=10, ha='left', va='top', color='red', fontweight='bold')

# Add reference lines
ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')
ax.axvline(0, color='grey', linewidth=0.5, linestyle='--')

# Set labels and title
ax.set_xlabel(f"Component 1 ({ax1})") # Formatting percentage if applicable
ax.set_ylabel(f"Component 2 ({ax2})")
ax.set_title("MCA – Axis 1 & 2 (Active vs Illustrative)")

# Add legend to distinguish groups
ax.legend(loc='best')

# Save the figure
img_address='doc_images/mca_prince_with_clusters.png'
plt.savefig(img_address, dpi=150, bbox_inches='tight')

plt.tight_layout()
plt.show()

### Single Most Representative Person

This is a different way of interpreting the clusters: find the person that is at the center of the MCA coordinates in the cluster. It is threfore somehow representative of the cluster

In [ ]:


# --- 1. SETUP (Assuming you already have these) ---
# dfa: DataFrame with MCA coordinates (numeric)
# cluster_labels: The result from KMeans.fit_predict()
# centroids: The result from kmeans.cluster_centers_


# Store results here
medoid_results = []

# --- 2. LOOP THROUGH EACH CLUSTER ---
unique_clusters = sorted(df_actives['cluster'].unique())

for k in unique_clusters:
    # A. Filter data: Get ONLY rows belonging to this specific cluster
    cluster_mask = df_actives['cluster'] == k
    cluster_coords = X_mca.iloc[:, :4][cluster_mask]
    
    # B. Get the specific centroid for this cluster
    # centroids is a numpy array, so we take row k
    current_centroid = numeric_centroids.iloc[k].values.reshape(1, -1) # Reshape to 2D for cdist
    
    # C. Calculate distances: Members vs. Their Own Centroid
    # We exclude the 'Cluster' column from calculation if it exists in cluster_members
    coords_only = X_mca.iloc[:, :-1] # Assumes last col is 'Cluster'
    
    distances = cdist(cluster_coords, current_centroid, metric='euclidean')
    
    # D. Find the index of the minimum distance within this subset
    min_dist_idx = distances.argmin()
    
    # E. Retrieve the ORIGINAL dataframe index of this person
    # cluster_members.index[min_dist_idx] gives the real ID from the original df
    original_index = X_mca.index[min_dist_idx]
    
    # F. Save results
    medoid_results.append({
        'Cluster_ID': k,
        'Medoid_Original_Index': original_index,
        'Min_Distance': distances[min_dist_idx][0]
    })

# --- 3. CREATE RESULT TABLE ---
medoid_df = pd.DataFrame(medoid_results)

# --- 4. DISPLAY WITH CATEGORIES ---
# Merge back with original data to see the actual categories (Age, City, etc.)
# Replace 'original_df' with the name of your raw categorical dataframe
medoids = medoid_df.merge(df_actives, left_on='Medoid_Original_Index', right_index=True, how='left')

print("Single Most Representative Person (Medoid) per Cluster:")
medoids

#### Inspect frequency of individuals identical with centroid

In [ ]:
### this dataframe contains only rows that are identical with the centroid in each cluster
merged_df = df_actives.merge(medoids, on=['gender', 'country', 'occup_m', 'occup_s', 'empl', 'cluster'], how='inner')

In [ ]:
### Persons with medoid value in proportion to population

# only persons identical to centroids of their cluster
print(len(merged_df))

# all the persons
print(len(df_pm))


# frequency
print(str(round((len(merged_df)/len(df_pm)*100),1))+' %')

#### Inspect with identical cluster profile (modes)

In [ ]:
clu_prof=cluster_profiles.reset_index(names='cluster')
print(len(clu_prof))
clu_prof.head()

In [ ]:
print(len(df_actives))

In [ ]:
### this dataframe contains only rows that are identical with the centroid in each cluster
merged_df_prof = df_actives.merge(clu_prof, on=['gender', 'country', 'occup_m', 'occup_s', 'empl', 'cluster'], how='inner')

In [ ]:
### Persons with medoid value in proportion to population

# only persons identical to centroids of their cluster
print(len(merged_df_prof))

# all the persons
print(len(df_pm))
cluster_profiles

# frequency
print(str(round((len(merged_df_prof)/len(df_pm)*100),1))+' %')



#### Keep the clu_prof 

We keep the first methodology to define the cluster center, based on modes.

But we observe from different runs for 16, 32 and 64 cluster, that smalle clusters tend to be nearer to the mean profile (vs the mode)

In [ ]:
clu_prof.head()

In [ ]:
print(len(merged_df_prof))
merged_df_prof.head(2)

In [ ]:
### Count per cluster how many rows have same values as centroid
dfcen = pd.DataFrame(merged_df_prof.groupby(by=['cluster']).size())
dfcen.columns=['n_cluster']
dfcen.sort_values(by='n_cluster', ascending=False).head()

In [ ]:
### How many rows in total have gender:female
dfgf = pd.DataFrame(df_actives[df_actives.gender=='female'].groupby(by=['cluster']).size())
dfgf.columns=['number_f']
(print(dfgf.sort_values(by='number_f', ascending=False).iloc[:30]))

In [ ]:
### Add cluster to df_pm
df_pm=df_pm.join(df_actives, rsuffix='_r')

In [ ]:
df_pm.head(2)

In [ ]:
### Group and count countries in clusters

# countries per cluster
result_df = pd.DataFrame(df_pm.groupby(by=['cluster', 'country_r']).size()).reset_index()
result_df.columns=['cluster','country_r','number']
result_df=result_df.sort_values(['cluster','number'], ascending=[True,False])
result_df.head()


# aggregated countries per cluster
dfg_country = result_df.groupby('cluster')[['country_r', 'number']].apply(
    lambda x: x.values.tolist()
).reset_index(name='aggregated_data')

### Take the first five per cluster
dfg_country['countries_list'] = dfg_country.aggregated_data.apply(
    lambda lst: ', '.join([f"{item[0]}: {item[1]}" for item in lst[:5]])
)
dfg_country=dfg_country.drop(columns=['cluster','aggregated_data'])


with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(dfg_country.head())

In [ ]:
### Group and count occupations in clusters

# countries per cluster
result_df = pd.DataFrame(df_pm.groupby(by=['cluster', 'occup_m']).size()).reset_index()
result_df.columns=['cluster','occup_m','number']
result_df=result_df.sort_values(['cluster','number'], ascending=[True,False])


# aggregated occupations per cluster
dfg_occupation = result_df.groupby('cluster')[['occup_m', 'number']].apply(
    lambda x: x.values.tolist()
).reset_index(name='aggregated_data')

### Take the first five per cluster
dfg_occupation['occ_list'] = dfg_occupation['aggregated_data'].apply(
    lambda lst: ', '.join([f"{item[0]}: {item[1]}" for item in lst[:5]])
)
dfg_occupation=dfg_occupation.drop(columns=['cluster','aggregated_data'])

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(dfg_occupation.head())

In [ ]:
### Group and count employer classes in clusters

# countries per cluster
result_df = pd.DataFrame(df_pm.groupby(by=['cluster', 'empl']).size()).reset_index()
result_df.columns=['cluster','empl','number']
result_df=result_df.sort_values(['cluster','number'], ascending=[True,False])


# aggregated occupations per cluster
dfg_empl = result_df.groupby('cluster')[['empl', 'number']].apply(
    lambda x: x.values.tolist()
).reset_index(name='aggregated_data')

### Take the first five per cluster
dfg_empl['empl_list'] = dfg_empl['aggregated_data'].apply(
    lambda lst: ', '.join([f"{item[0]}: {item[1]}" for item in lst[:5]])
)
dfg_empl=dfg_empl.drop(columns=['cluster','aggregated_data'])

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(dfg_empl.head())


#### Join and inspect

Aggregating the former tables and counts allows to inspect the clusters and identifying the structuring elements of the clusters 

In [ ]:
clu_prof.iloc[:,[1,2,3,4,5,6,11]].head()

In [ ]:
clu_prof_short=clu_prof.iloc[:,[1,2,3,4,5,6,11]]

In [ ]:
print(len(clu_prof_short))
clu_prof_short.iloc[:,[5,4]].sort_values('number_in_cl', ascending=False)

In [ ]:
### Join the different dataframes using the default joining on indexes
# If an error appears in joining, restart the whole process from cenroid_df onward
if 'number_f' in clu_prof_short.columns:
    clu_prof_short = clu_prof_short.drop(columns=['number_f'])

clu_prof_short=clu_prof_short.join(dfcen)
clu_prof_short.rename(columns={'n_cluster': 'number_centroid'}, inplace=True)
clu_prof_short=clu_prof_short.join(dfgf).fillna(0)
clu_prof_short['number_f'] = clu_prof_short['number_f'].astype(int)
clu_prof_short['prop_f'] = clu_prof_short.apply(lambda x: (x['number_f']/x['number_in_cl']), axis=1)
clu_prof_short['prop_f']=clu_prof_short['prop_f'].round(2)
clu_prof_short=clu_prof_short.join(dfg_country)
clu_prof_short=clu_prof_short.join(dfg_occupation)
clu_prof_short=clu_prof_short.join(dfg_empl)

In [ ]:
clu_prof_short.number_centroid=clu_prof_short.number_centroid.astype('int')

In [ ]:
print(clu_prof_short.columns.to_list())

In [ ]:
clu_prof_short['prop_centr']=clu_prof_short.apply(lambda x : x['number_centroid']/x['number_in_cl'], axis=1).round(2)

In [ ]:
clu_prof_short=clu_prof_short[['gender', 'country', 'occup_m', 'occup_s','empl', 'number_in_cl', 
                              'number_centroid', 'prop_centr',
                              'number_f', 'prop_f', 'label',
                              'countries_list', 'occ_list', 
                               'empl_list']]

In [ ]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(clu_prof_short.sort_values(by='number_in_cl', ascending=False).style.set_properties(subset=['occ_list', 'countries_list'], **{'text-align': 'left'}))

In [ ]:
### write centroids to database table for more deep inspection
db = '../../data_analysis.db'

#table_name='clusters_kmodes_centroids_c54'
table_name='mca_kmeans_clusters_centroids'

#clu_prof_short['run']='cen16'
#clu_prof_short['run']='cen32'
clu_prof_short['run']='cen54'
#clu_prof_short['run']='cen64'


conn = sql.connect(db)
# commented for safety, uncomment to execute
clu_prof_short.to_sql(table_name, conn, if_exists='append', index=True)
conn.close()

In [ ]:
pcf=df_pm[['person_uri', 'per_activ','gender','country', 'empl','occup_m', 'occup_s']]
print(len(pcf))
pcf.head()

In [ ]:
### write centroids to database table for more deep inspection
db = '../../data_analysis.db'

table_name='clusters_kmodes_centroids_c54'
table_name='person_coded_features'



conn = sql.connect(db)
# commented for safety, uncomment to execute
pcf.to_sql(table_name, conn, if_exists='append', index=True)
conn.close()

## Test if relation between clusters and periods

In [ ]:
# Validate against Generation (Illustrative Variable)
# Cross-tabulation : contingency_table

df = df_pm

observed = pd.crosstab(df['cluster'], df['per_activ'])


In [ ]:
bl.check_chi_square_test_validity(observed)

In [ ]:
expected=bl.bivariate_stats(observed)

### CA

In [ ]:
afc = fa.CA(row_labels=observed.index,col_labels=observed.columns)
afc.fit(observed.values)

In [ ]:
### Inertia (Phi-square - Eigenvalue):  0.108
cal.print_eigenvalue(afc)

In [ ]:
cal.dim_contributions(afc)

In [ ]:
# Represent dimension 1 and 2
afc.mapping(num_x_axis=1,num_y_axis=2,figsize=(8,8))

In [ ]:
# Represent dimension 2 and 3
afc.mapping(num_x_axis=3,num_y_axis=4,figsize=(8,8))

In [ ]:
### Number of clusters
width= df['cluster'].max()
pp = bl.plot_chi2_residuals(observed.T, figsize=(width, 10))

In [ ]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(pd.DataFrame(clu_prof_short[['label','number_in_cl','number_centroid','number_f', 'prop_f']]))

In [ ]:
pers_cluster=df_pm[['person_uri', 'cluster']]
pers_cluster.head()

In [ ]:
### write clusters to database table for more deep inspection
db = '../../data_analysis.db'

table_name='mca_kmeans_clusters'

# pers_cluster['run']='cen16'
# pers_cluster['run']='cen32'
pers_cluster['run']='cen54'
# pers_cluster['run']='cen64'

conn = sql.connect(db)
# cursor = conn.cursor()
pers_cluster.to_sql(table_name, conn, if_exists='append', index=True)
conn.close()

## Plot clusters

In [ ]:
categorical_cols = df_actives.columns.to_list()[:4]
print(categorical_cols)

In [ ]:
df_actives.head(1)

In [ ]:
## ici on voit les attractions de modalités
# les variables au centre ou très liées structurent le cluster
pict_address='images/kmeans_ACM_clusters_4_variables_32cl.png'
cf.plot_cluster_networks(df_actives, categorical_cols,32, 
        pict_address=pict_address)